Setup & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
processed_path = root / "data" / "processed" / "features_with_models.csv"
if not processed_path.exists():
    raise FileNotFoundError(f"Modeled features not found: {processed_path}")

df = pd.read_csv(processed_path)
df["window_start"] = pd.to_datetime(df["window_start"])
df["hour"] = df["window_start"].dt.hour + df["window_start"].dt.minute/60
df["dow"] = df["window_start"].dt.dayofweek
df.head()

What sensors exist as features?

In [ ]:
ignore_cols = {"window_start","cluster","is_anomaly","anomaly_score","hour","dow"}
non_sensor_cols = {"total_events","n_sensors_active","tod_sin","tod_cos","is_inactive"}
sensor_cols = [c for c in df.columns if c not in ignore_cols and c not in non_sensor_cols]
len(sensor_cols), sensor_cols[:10]

Cluster size and time-of-day profile

In [ ]:
cluster_counts = df["cluster"].value_counts().sort_index()
cluster_counts

In [ ]:
# Time-of-day distribution per cluster
for k in sorted(df["cluster"].unique()):
    hours = df.loc[df["cluster"]==k, "hour"].values
    plt.figure()
    plt.hist(hours, bins=24)
    plt.title(f"Cluster {k}: hour-of-day distribution")
    plt.xlabel("Hour")
    plt.ylabel("Count")
    plt.show()

Top sensors per cluster

In [ ]:
def top_sensors_for_cluster(k, topn=10):
    sub = df[df["cluster"]==k]
    active_sub = sub[sub["total_events"] > 0] if "total_events" in sub.columns else sub
    ref = active_sub if not active_sub.empty else sub
    means = ref[sensor_cols].mean().sort_values(ascending=False)
    return means.head(topn)

for k in sorted(df["cluster"].unique()):
    print(f"\n=== Cluster {k} top sensors ===")
    print(top_sensors_for_cluster(k, topn=12))

Cluster summary table you can paste into report

In [ ]:
summary_rows = []
for k in sorted(df["cluster"].unique()):
    sub = df[df["cluster"]==k]
    top_sensors = top_sensors_for_cluster(k, topn=2)
    row = {
        "cluster": k,
        "n_windows": len(sub),
        "mean_total_events": sub["total_events"].mean(),
        "mean_unique_sensors": sub["n_sensors_active"].mean(),
        "inactive_fraction": (sub["total_events"] == 0).mean(),
        "peak_hour": sub["hour"].round().mode().iloc[0] if len(sub) else np.nan,
        "top_sensor_1": top_sensors.index[0] if len(top_sensors) > 0 else "",
        "top_sensor_2": top_sensors.index[1] if len(top_sensors) > 1 else "",
    }
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).sort_values("cluster")
summary

Assign human-readable routine labels

In [ ]:
# Fill this after inspecting time histograms + top sensors
cluster_name = {
    0: "Night inactivity",
    1: "Morning kitchen routine",
    2: "Evening routine",
    # ...
}

df["routine_label"] = df["cluster"].map(cluster_name).fillna("Unlabeled routine")
df[["window_start","cluster","routine_label","total_events","n_sensors_active"]].head(20)